# Resting EEG Feature Extraction

This notebook turns the cleaned resting-state EEG epochs from `03_Batch_Preprocessing.ipynb` into one feature table for downstream modelling or statistical analysis.

The implementation details live in `src/eeg_feature_extraction.py`. This notebook is organised as a walkthrough: first we check the input data, then inspect what each feature family means, then run extraction and save the final table.

## 1. Setup

This cell sets paths, imports the feature-extraction helpers, and keeps the notebook connected to the local `src/` package.

In [14]:
from importlib import reload
from pathlib import Path

import os
import sys

import mne
import numpy as np
import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
if not (project_root / "src").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

os.environ.setdefault("MPLCONFIGDIR", str(project_root / ".matplotlib"))
mne.set_log_level("WARNING")

import src.eeg_feature_extraction as efe

reload(efe)

processed_eeg_dir = project_root / "data" / "processed" / "eeg"
features_dir = project_root / "data" / "features"
article_output_dir = project_root / "reports" / "feature_extraction_article"
article_output_dir.mkdir(parents=True, exist_ok=True)

print(f"Project root: {project_root}")
print(f"Processed EEG dir: {processed_eeg_dir}")
print(f"Features dir: {features_dir}")

Project root: /Users/nataliemarryatt/neurogenetics-ml
Processed EEG dir: /Users/nataliemarryatt/neurogenetics-ml/data/processed/eeg
Features dir: /Users/nataliemarryatt/neurogenetics-ml/data/features


## 2. Locate and Inspect Clean Resting EEG Epochs

The preprocessing notebook saves one cleaned epoch file per subject and condition:

- `eyes_open`: resting EEG recorded with eyes open
- `eyes_closed`: resting EEG recorded with eyes closed

Each row below is one feature-extraction input. Later, each row will become one row in the feature table.

Before calculating features, check that each file has a reasonable number of clean epochs and EEG channels.

What to look at:
- `n_epochs`: how many clean fixed-length segments survived preprocessing and rejection
- `n_channels`: how many usable EEG channels remain after excluding bad channels
- `duration_minutes`: approximate amount of clean EEG contributing to the features

In [15]:
epoch_index = efe.find_clean_epoch_files(processed_eeg_dir)

print(f"Found {len(epoch_index)} clean epoch files across {epoch_index['subject_id'].nunique() if not epoch_index.empty else 0} subjects.")
display(epoch_index)

epoch_summary = efe.summarize_epoch_files(epoch_index)
display(epoch_summary)

Found 156 clean epoch files across 78 subjects.


,subject_id,condition,path
0,sub-01,eyes_closed,/Users/nataliemarryatt/neurogenetics-ml/data/processed/eeg/sub-01/eeg/sub-01_task-rest_eyes_closed_clean-epo.fif
1,sub-01,eyes_open,/Users/nataliemarryatt/neurogenetics-ml/data/processed/eeg/sub-01/eeg/sub-01_task-rest_eyes_open_clean-epo.fif
2,sub-02,eyes_closed,/Users/nataliemarryatt/neurogenetics-ml/data/processed/eeg/sub-02/eeg/sub-02_task-rest_eyes_closed_clean-epo.fif
3,sub-02,eyes_open,/Users/nataliemarryatt/neurogenetics-ml/data/processed/eeg/sub-02/eeg/sub-02_task-rest_eyes_open_clean-epo.fif
4,sub-03,eyes_closed,/Users/nataliemarryatt/neurogenetics-ml/data/processed/eeg/sub-03/eeg/sub-03_task-rest_eyes_closed_clean-epo.fif
...,...,...,...
151,sub-78,eyes_open,/Users/nataliemarryatt/neurogenetics-ml/data/processed/eeg/sub-78/eeg/sub-78_task-rest_eyes_open_clean-epo.fif
152,sub-79,eyes_closed,/Users/nataliemarryatt/neurogenetics-ml/data/processed/eeg/sub-79/eeg/sub-79_task-rest_eyes_closed_clean-epo.fif
153,sub-79,eyes_open,/Users/nataliemarryatt/neurogenetics-ml/data/processed/eeg/sub-79/eeg/sub-79_task-rest_eyes_open_clean-epo.fif
154,sub-80,eyes_closed,/Users/nataliemarryatt/neurogenetics-ml/data/processed/eeg/sub-80/eeg/sub-80_task-rest_eyes_closed_clean-epo.fif


,subject_id,condition,n_epochs,n_channels,sfreq,duration_minutes
0,sub-01,eyes_closed,179,122,250.0,5.954733
1,sub-01,eyes_open,119,124,250.0,3.958733
2,sub-02,eyes_closed,179,126,250.0,5.954733
3,sub-02,eyes_open,118,125,250.0,3.925467
4,sub-03,eyes_closed,180,126,250.0,5.988000
...,...,...,...,...,...,...
151,sub-78,eyes_open,119,126,250.0,3.958733
152,sub-79,eyes_closed,179,122,250.0,5.954733
153,sub-79,eyes_open,119,125,250.0,3.958733
154,sub-80,eyes_closed,180,127,250.0,5.988000


## 3. Define Feature Settings

These are the main analysis choices used by the helper functions later in the notebook.

The regional mapping uses compact channel-name prefix rules for the high-density EEG montage. This keeps the definition readable while still assigning all available EEG channels to broad scalp regions.

In [16]:
EEG_BANDS = {
    "delta": (1.0, 4.0),
    "theta": (4.0, 8.0),
    "alpha": (8.0, 13.0),
    "beta": (13.0, 30.0),
    "gamma_low": (30.0, 40.0),
}

# Delta connectivity is skipped because 2-second epochs are too short for reliable 1 Hz connectivity estimates.
CONNECTIVITY_BANDS = {
    "theta": EEG_BANDS["theta"],
    "alpha": EEG_BANDS["alpha"],
    "beta": EEG_BANDS["beta"],
    "gamma_low": EEG_BANDS["gamma_low"],
}

REGION_PREFIXES = {
    "frontal": ("Fp", "AF", "AFF", "F1", "F2", "F3", "F4", "F5", "F6", "F7", "F8", "F9", "F10", "Fz", "FFC", "FFT"),
    "central": ("FC", "FCC", "C1", "C2", "C3", "C4", "C5", "C6", "Cz", "CCP"),
    "temporal": ("FT", "FTT", "T7", "T8", "TTP", "TP", "TPP"),
    "parietal": ("CP", "CPP", "P1", "P2", "P3", "P4", "P5", "P6", "P7", "P8", "P9", "P10", "Pz", "PPO"),
    "occipital": ("PO", "POO", "O", "OI"),
}

all_eeg_channels = sorted(
    {
        ch
        for path in epoch_index["path"]
        for ch in efe.load_epochs(path).copy().pick("eeg", exclude=[]).ch_names
    }
)

REGIONS = efe.make_regions_from_prefixes(all_eeg_channels, REGION_PREFIXES)

# Make these notebook-level choices available to the reusable helper functions.
efe.EEG_BANDS = EEG_BANDS
efe.CONNECTIVITY_BANDS = CONNECTIVITY_BANDS
efe.REGIONS = REGIONS

pd.Series({region: len(channels) for region, channels in REGIONS.items()}, name="n_channels").to_frame()

,n_channels
frontal,34
central,25
temporal,20
parietal,30
occipital,18


## 4. Inspect One Example Participant

Before running the whole cohort, extract features from one file. This makes the output easier to understand.

The feature values below are not interpreted as group effects. They are a sanity check showing the structure of one subject-condition row.

In [18]:
# take first subject file and load epochs prepared in preprocessing workflow
example_row = epoch_index.iloc[0]
example_epochs = efe.load_epochs(example_row["path"])

print(f"Example: {example_row['subject_id']} | {example_row['condition']}")
print(f"Epochs: {len(example_epochs)}")
print(f"Sampling frequency: {example_epochs.info['sfreq']} Hz")

Example: sub-01 | eyes_closed
Epochs: 179
Sampling frequency: 250.0 Hz


### 4.1 Spectral Features

Spectral features summarise power in frequency bands. For resting EEG, these are usually the most central features.

For this one example participant, the code below shows the main calculation: estimate the power spectral density, integrate power within a band, then summarise across epochs and channels. The reusable function in `src/eeg_feature_extraction.py` repeats this same logic for all bands and regions.

Metrics calculated in this example:
- `alpha_absolute_power_global`: average alpha-band power across all clean epochs and good EEG channels
- `alpha_relative_power_global`: alpha power divided by total 1-40 Hz power, then averaged
- `n_epochs`: number of clean EEG epochs used
- `n_channels`: number of good EEG channels included
- `n_frequency_bins`: number of frequency points in the PSD estimate

The scalable function later repeats this logic for all EEG bands and regions.
This section also saves a clean PSD figure with the analysed frequency bands shaded for use in the article.


In [20]:
from scipy.integrate import simpson

# calculate how many samples to use for each Welch PSD window (250 freq * 2s window = 500 samples)
n_per_seg = int(round(example_epochs.info["sfreq"] * efe.WELCH_WINDOW_S))

# compute psd using Welch's method using 2 second windows
spectrum = example_epochs.compute_psd(
    method="welch",
    fmin=efe.PSD_FMIN,
    fmax=efe.PSD_FMAX,
    picks="eeg",
    exclude="bads",
    n_per_seg=n_per_seg,
    n_fft=n_per_seg,
)

psd = spectrum.get_data()  # shape: epochs x channels x frequencies
freqs = spectrum.freqs

# calculate alpha power
alpha_mask = (freqs >= 8) & (freqs < 13)
alpha_power = simpson(psd[..., alpha_mask], x=freqs[alpha_mask], axis=-1)

# calculate total power
total_mask = (freqs >= 1) & (freqs < 40)
total_power = simpson(psd[..., total_mask], x=freqs[total_mask], axis=-1)

# example of alpha spectral metrics
example_spectral_summary = pd.Series(
    {
        "alpha_absolute_power_global": alpha_power.mean(),
        "alpha_relative_power_global": (alpha_power / total_power).mean(),
        "n_epochs": psd.shape[0],
        "n_channels": psd.shape[1],
        "n_frequency_bins": psd.shape[2],
    }
)

example_spectral_summary.to_frame("value")

,value
alpha_absolute_power_global,1.218393e-11
alpha_relative_power_global,3.520549e-01
n_epochs,1.790000e+02
n_channels,1.220000e+02
n_frequency_bins,7.900000e+01


In [ ]:
mean_psd = psd.mean(axis=(0, 1))

psd_figure_path = efe.save_article_psd_band_figure(
    freqs,
    mean_psd,
    article_output_dir,
    label=f"{example_row['subject_id']} {example_row['condition']}",
)

example_spectral_summary.to_frame("value").to_csv(
    article_output_dir / "spectral_feature_example.tsv",
    sep="\t",
)
print(f"Saved PSD figure to: {psd_figure_path}")


### 4.2 Aperiodic Features

Aperiodic features separate the broadband 1/f-like background from narrow oscillatory peaks. This is important because an apparent bandpower difference may reflect a broad spectral slope change rather than a true change in a rhythmic oscillation.

For one participant, the important steps are: average the PSD across epochs and channels, fit a spectral model, then pull out the fitted background slope and peak parameters.

Metrics calculated in this example:
- `aperiodic_offset`: overall vertical position of the fitted background spectrum
- `aperiodic_exponent`: steepness of the spectral slope
- `n_oscillatory_peaks`: number of fitted rhythmic peaks

The scalable function also saves strongest-peak frequency, power, and bandwidth when peaks are detected.
This section also saves the fitted `specparam` plot so the separation between the aperiodic background and oscillatory peaks can be shown visually.


In [23]:
from specparam import SpectralModel

# calculate mean psd across across all epochs and channels
mean_psd = psd.mean(axis=(0, 1))

# create and fit a spectral parameterisation model to subjects average PSD
spectral_model = SpectralModel(
    peak_width_limits=(1.0, 8.0),
    max_n_peaks=6,
    verbose=False,
)
spectral_model.fit(freqs, mean_psd, [efe.PSD_FMIN, efe.PSD_FMAX])

# split fitted spectrum into aperiodic background features and detected oscillitory peaks
param_dict = spectral_model.results.params.asdict()
aperiodic_params = param_dict["aperiodic_fit"]
peak_params = param_dict["peak_fit"]

example_aperiodic_summary = pd.Series(
    {
        "aperiodic_offset": aperiodic_params[0],
        "aperiodic_exponent": aperiodic_params[1],
        "n_oscillatory_peaks": len(peak_params),
    }
)

example_aperiodic_summary.to_frame("value")

,value
aperiodic_offset,-11.351893
aperiodic_exponent,0.964846
n_oscillatory_peaks,2.000000


In [ ]:
aperiodic_figure_path = efe.save_article_specparam_figure(
    spectral_model,
    article_output_dir,
)

aperiodic_feature_table = example_aperiodic_summary.to_frame("value")
aperiodic_feature_table.to_csv(
    article_output_dir / "aperiodic_feature_example.tsv",
    sep="\t",
)
print(f"Saved aperiodic figure to: {aperiodic_figure_path}")
display(aperiodic_feature_table)


### 4.3 Time-Domain and Complexity Features

These features describe waveform shape and signal irregularity. They are useful as compact signal summaries, but should be interpreted cautiously because residual artifacts can also influence them.

For one participant, the main operation is to work directly with the cleaned EEG array: epochs x channels x time points. Hjorth features use the signal variance and the variance of its first and second differences.

Metrics calculated in this example:
- `signal_sd_uv`: overall standard deviation of the EEG signal in microvolts
- `mean_peak_to_peak_uv`: average within-epoch amplitude range
- `hjorth_activity`: variance of the signal
- `hjorth_mobility`: how quickly the signal changes
- `hjorth_complexity`: how much the signal shape deviates from a simple sine-like waveform
- `permutation_entropy_first_channel`: irregularity of the first channel's epoch-averaged signal

The scalable function uses related names such as `td_sd_uv`, `td_peak_to_peak_uv`, and additional complexity summaries.
This section also saves one cleaned 2-second EEG trace to show the time-domain signal these metrics are calculated from.


In [25]:
import antropy as ant

# load cleaned EEG data as an array
data_uv = example_epochs.copy().pick("eeg", exclude="bads").get_data() * 1e6

# calcualte how signal changes over time, used for Hjorth features
first_diff = np.diff(data_uv, axis=-1) # difference between consecutive timepoints
second_diff = np.diff(first_diff, axis=-1) # difference between first differences

# calculate hjorth features
activity = np.var(data_uv, axis=-1) 
mobility = np.sqrt(np.var(first_diff, axis=-1) / activity)
complexity = np.sqrt(np.var(second_diff, axis=-1) / np.var(first_diff, axis=-1)) / mobility

 # average across epochs, then use the first EEG channel as an example signal for permutation entropy
first_channel_epoch_average = data_uv.mean(axis=0)[0]

example_time_complexity_summary = pd.Series(
    {
        "signal_sd_uv": data_uv.std(),
        "mean_peak_to_peak_uv": np.ptp(data_uv, axis=-1).mean(),
        "hjorth_activity": np.nanmean(activity),
        "hjorth_mobility": np.nanmean(mobility),
        "hjorth_complexity": np.nanmean(complexity),
        "permutation_entropy_first_channel": ant.perm_entropy(first_channel_epoch_average, normalize=True),
    }
)

example_time_complexity_summary.to_frame("value")

,value
signal_sd_uv,5.944253
mean_peak_to_peak_uv,32.099611
hjorth_activity,35.145509
hjorth_mobility,0.337091
hjorth_complexity,1.953355
permutation_entropy_first_channel,0.790996


In [ ]:
epoch_trace_path = efe.save_article_epoch_trace_figure(
    example_epochs,
    article_output_dir,
)

example_time_complexity_summary.to_frame("value").to_csv(
    article_output_dir / "time_complexity_feature_example.tsv",
    sep="\t",
)
print(f"Saved time-domain trace to: {epoch_trace_path}")


### 5.5 Connectivity and Graph Features

Connectivity features summarise synchronisation between channels. Here the default is wPLI, which focuses on phase-lagged coupling and is less sensitive to zero-lag volume conduction than ordinary coherence.

Because the current preprocessing uses 2-second epochs, this notebook estimates connectivity from theta upward. Delta connectivity would need longer epochs to contain enough low-frequency cycles.

For one participant, the example below calculates an alpha-band wPLI matrix. Each cell in this matrix is the alpha-band phase-lagged connectivity value between two EEG channels. The upper triangle of the matrix is then summarised to describe overall connectivity strength.

Graph metrics are calculated by thresholding the wPLI matrix and treating the remaining strongest connections as a network:
- nodes = EEG channels
- edges = retained wPLI connections

Metrics calculated in this example:
- `alpha_wpli_mean`: average alpha-band phase-lagged coupling across channel pairs
- `alpha_wpli_sd`: variability of alpha-band wPLI across channel pairs
- `alpha_graph_density`: proportion of retained network edges after thresholding
- `alpha_graph_degree_mean`: average number of retained connections per channel
- `alpha_graph_strength_mean`: average weighted connectivity strength per channel
- `alpha_graph_clustering_mean`: tendency for connected channels to form local groups
- `alpha_graph_path_length`: average shortest path through the weighted network
- `alpha_graph_global_efficiency`: efficiency of information transfer across the whole network
- `alpha_graph_local_efficiency`: efficiency within local neighbourhoods
- `alpha_graph_small_world_sigma`: small-world estimate comparing clustering and path length with matched random graphs

The scalable function repeats this for theta, alpha, beta, and low gamma.

This section also saves the alpha-band wPLI matrix as the article figure for connectivity and graph features.


In [27]:
from mne_connectivity import spectral_connectivity_epochs

# start from cleaned EEG epochs and calculate alpha-band wPLI between every channel pair
eeg_epochs = example_epochs.copy().pick("eeg", exclude="bads")

alpha_connectivity = spectral_connectivity_epochs(
    eeg_epochs,
    method="wpli",
    mode="multitaper",
    sfreq=example_epochs.info["sfreq"],
    fmin=8,
    fmax=13,
    faverage=True,
    verbose=False,
)

# convert the result into a channel x channel connectivity matrix
matrix = alpha_connectivity.get_data(output="dense")[:, :, 0]
matrix = np.maximum(matrix, matrix.T)
np.fill_diagonal(matrix, 0.0)

# summarise all unique channel-pair connections
upper_triangle = matrix[np.triu_indices_from(matrix, k=1)]

# threshold the matrix so the strongest 25% of connections form the graph
threshold = np.nanpercentile(upper_triangle, 75)
adjacency = (matrix >= threshold).astype(float)
np.fill_diagonal(adjacency, 0.0)
weighted_adjacency = np.where(adjacency, matrix, 0.0)

# basic graph summaries
density = adjacency.sum() / (adjacency.shape[0] * (adjacency.shape[0] - 1))
degree = adjacency.sum(axis=1)
strength = weighted_adjacency.sum(axis=1)
triangles = np.diag(adjacency @ adjacency @ adjacency) / 2
degree_denominator = degree * (degree - 1)
clustering = np.divide(
    2 * triangles,
    degree_denominator,
    out=np.full_like(triangles, np.nan, dtype=float),
    where=degree_denominator > 0,
)
clustering_mean = np.nanmean(clustering)

# longer graph summaries are handled by the reusable feature-extraction helpers
integration = efe._graph_integration_metrics(weighted_adjacency)
local_efficiency = efe._local_efficiency(adjacency, weighted_adjacency)
small_world = efe._small_world_metrics(adjacency, clustering_mean)

example_connectivity_summary = pd.Series(
    {
        "alpha_wpli_mean": np.nanmean(upper_triangle),
        "alpha_wpli_sd": np.nanstd(upper_triangle),
        "alpha_graph_density": density,
        "alpha_graph_degree_mean": np.mean(degree),
        "alpha_graph_strength_mean": np.mean(strength),
        "alpha_graph_clustering_mean": clustering_mean,
        "alpha_graph_path_length": integration["path_length"],
        "alpha_graph_global_efficiency": integration["global_efficiency"],
        "alpha_graph_local_efficiency": local_efficiency,
        "alpha_graph_small_world_sigma": small_world["small_world_sigma"],
        "n_channels_in_matrix": matrix.shape[0],
    }
)

example_connectivity_summary.to_frame("value")


,value
alpha_wpli_mean,0.338673
alpha_wpli_sd,0.157417
alpha_graph_density,0.250102
alpha_graph_degree_mean,30.262295
alpha_graph_strength_mean,16.847450
alpha_graph_clustering_mean,0.452656
alpha_graph_path_length,3.409022
alpha_graph_global_efficiency,0.310676
alpha_graph_local_efficiency,0.395507
alpha_graph_small_world_sigma,1.660778


In [ ]:
connectivity_figure_path = efe.save_article_connectivity_matrix_figure(
    matrix,
    article_output_dir,
)

example_connectivity_summary.to_frame("value").to_csv(
    article_output_dir / "connectivity_graph_feature_example.tsv",
    sep="\t",
)
print(f"Saved connectivity figure to: {connectivity_figure_path}")
display(example_connectivity_summary.to_frame("value"))


### 5.6 Microstate Features

Microstates are relevant for resting EEG, but they need special handling in a machine-learning workflow. The templates should be learned only from training subjects, then applied to validation/test subjects. That avoids leaking information from held-out participants into the feature extraction step.

For learning purposes, I did a small 5-subject microstate pilot. This pilot fits four microstate templates using GFP peaks from the first five eyes-closed subjects. It is intended for my learning and timing estimate, not as the final leakage-safe microstate feature extraction workflow.

For modelling, fit microstate templates after train/test splitting using training subjects only, then backfit those templates to validation/test subjects.
The code below runs a small five-subject eyes-closed pilot to produce an article figure and estimate runtime. This is not the final leakage-safe microstate feature extraction step.


In [ ]:
# This pilot is for an article figure and timing estimate, not final modelling features.
# Final microstate templates should be fitted after train/test splitting.
microstate_summary = efe.run_microstate_article_pilot(
    epoch_index,
    article_output_dir,
    n_subjects=5,
    condition="eyes_closed",
    n_clusters=4,
    n_init=10,
    random_state=42,
)

microstate_summary.loc[
    [
        "n_subjects",
        "n_epochs_total",
        "n_common_channels",
        "n_gfp_peaks",
        "total_seconds",
        "total_minutes",
        "global_explained_variance",
    ]
].to_frame("value")


## Final Note

This notebook is for understanding and demonstrating the feature extraction workflow on example data. Batch feature extraction across all available subjects, with subject-level train/test split preparation for modelling, is handled in `05_Batch_Feature_Extraction.ipynb`.
